In [ ]:
# made by Eli (lili041 --Github) with Google Gemini
# you have to fill in the  -path of .json textcritics-  and the  -path or .svg folder- !

# ACHTUNG: TODO werden übersprungen, das heisst aber auch, dass sie innerhalb eines Blockes nicht einberechnet werden, 
# und danach weitergezählt wird, als ob nichts wäre: g-tkk-1, g-tkk-2, g-tkk-3, g-tkk-4, TODO, g-tkk-5, g-tkk-6, ...


# --- KONFIGURATION ---


##### fill in:
json_path = '/Users/Elias/awg-app/src/assets/data/edition/series/1/section/5/op4/textcritics.json'

##### fill in:
svg_folder = '/Users/Elias/awg-app/src/assets/img/edition/series/1/section/5/op4' 


prefix = "g-tkk-"


import json
import os
import re

def extract_numbers(text):
    """Extrahiert Ziffern (z.B. 'M_143' -> '143')"""
    return "".join(re.findall(r'\d+', str(text)))

def display_uncertainties(data, prefix, loaded_svgs):
    """Checks JSON and SVG files for IDs that don't match the new prefix."""
    print(f"\n--- UNCERTAINTY & ERROR REPORT ---")
    errors_found = 0
    
    all_entries = data.get('textcritics', data) if isinstance(data, dict) else data
    for entry in all_entries:
        entry_id = entry.get('id', 'Unknown')
        comments_list = entry.get('commentary', {}).get('comments', [])
        for comment_group in comments_list:
            for b_comment in comment_group.get('blockComments', []):
                val = b_comment.get('svgGroupId')
                if val and not val.startswith(prefix) and val != "TODO":
                    print(f"  [!] JSON ERROR: Unchanged ID '{val}' in Entry: {entry_id}")
                    errors_found += 1

    tkk_id_regex = re.compile(r'<[^>]+?class=["\']tkk["\'][^>]+?id=["\']([^"\']+)["\']|<[^>]+?id=["\']([^"\']+)["\'][^>]+?class=["\']tkk["\']')
    
    for filename, sdata in loaded_svgs.items():
        matches = tkk_id_regex.findall(sdata["content"])
        for match in matches:
            found_id = match[0] if match[0] else match[1]
            if not found_id.startswith(prefix):
                print(f"  [!] SVG ORPHAN: ID '{found_id}' with class 'tkk' in {filename} was NOT updated.")
                errors_found += 1
    
    if errors_found == 0:
        print("  [✓] All JSON and SVG 'tkk' IDs successfully updated.")
    else:
        print(f"  [!] Total issues found: {errors_found}")

# 1. Daten laden
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

all_entries = data.get('textcritics', data) if isinstance(data, dict) else data
all_svg_files = [f for f in os.listdir(svg_folder) if f.endswith('.svg')]

final_svg_cache = {}
loaded_svg_texts = {}

def get_svg_text(filename):
    if filename not in loaded_svg_texts:
        path = os.path.join(svg_folder, filename)
        with open(path, 'r', encoding='utf-8') as f:
            content = f.read()
        loaded_svg_texts[filename] = {"content": content, "path": path}
        final_svg_cache[filename] = loaded_svg_texts[filename]
    return loaded_svg_texts[filename]

current_main_number = None
relevant_svgs_for_block = []
id_mapping = {}

print(f"--- Starte Bearbeitung mit SkRT-Speziallogik ---")

# 2. Iteration durch die JSON
for entry in all_entries:
    if not isinstance(entry, dict): continue

    new_id = entry.get('id', '')
    if new_id:
        for fname, sdata in loaded_svg_texts.items():
            with open(sdata['path'], 'w', encoding='utf-8') as f:
                f.write(sdata['content'])
        
        loaded_svg_texts.clear()
        id_mapping.clear()
        
        current_main_number = extract_numbers(new_id)
        
        if "SkRT" in new_id:
            relevant_svgs_for_block = [
                f for f in all_svg_files 
                if current_main_number in extract_numbers(f) and "Reihentabelle" in f
            ]
            print(f"\n SkRT-Anker erkannt: {new_id}")
        else:
            relevant_svgs_for_block = [
                f for f in all_svg_files 
                if current_main_number in extract_numbers(f) and "Reihentabelle" not in f
            ]
            print(f"\n Standard-Anker: {new_id}")
            
        print(f"   Zugeordnete SVGs: {relevant_svgs_for_block}")

    comments_list = entry.get('commentary', {}).get('comments', [])
    for comment_group in comments_list:
        for b_comment in comment_group.get('blockComments', []):
            old_val = b_comment.get('svgGroupId')
            if not old_val or old_val == "TODO": continue

            # Only match the ID if the same tag contains class="tkk"
            # This regex handles id before class OR class before id within the same < > block
            pattern = rf'(<[^>]+?id=["\']{re.escape(old_val)}["\'][^>]+?class=["\']tkk["\']|<[^>]+?class=["\']tkk["\'][^>]+?id=["\']{re.escape(old_val)}["\'][^>]*?>)'
            
            found_in_svg = None
            for svg_filename in relevant_svgs_for_block:
                svg_data = get_svg_text(svg_filename)
                if re.search(pattern, svg_data["content"]):
                    found_in_svg = svg_filename
                    break
            
            if found_in_svg:
                if old_val not in id_mapping:
                    id_mapping[old_val] = f"{prefix}{len(id_mapping) + 1}"
                
                new_val = id_mapping[old_val]
                
                print(f"    [JSON] Changing: '{old_val}' -> '{new_val}'")
                b_comment['svgGroupId'] = new_val
                
                svg_data = get_svg_text(found_in_svg)
                
                # Replace only the ID part within that specific matched tag
                # This sub-replacement ensures we don't accidentally mess up the class part
                def replace_id(match):
                    full_tag = match.group(0)
                    return full_tag.replace(f'id="{old_val}"', f'id="{new_val}"').replace(f"id='{old_val}'", f"id='{new_val}'")

                svg_data["content"] = re.sub(pattern, replace_id, svg_data["content"])
                print(f"    [SVG]  In {found_in_svg}: Updated ID for class 'tkk'")
            else:
                print(f"    [ERROR] ID '{old_val}' with class 'tkk' not found in relevant SVGs for {new_id}")

# 3. Abschluss-Speicherung
for fname, sdata in loaded_svg_texts.items():
    with open(sdata['path'], 'w', encoding='utf-8') as f:
        f.write(sdata['content'])

display_uncertainties(data, prefix, final_svg_cache)

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

print(f"\n Fertig!")

--- Starte Bearbeitung mit SkRT-Speziallogik ---

 Standard-Anker: op4_WE
   Zugeordnete SVGs: ['M140_Textfassung1-1von3-final.svg', 'M142_Textfassung2-2von2-final.svg', 'M138_Textfassung2-4von5-final.svg', 'M142_Textfassung2-1von2-final.svg', 'M140_Textfassung1-2von3-final.svg', 'M140_Textfassung1-3von3-final.svg', 'M138_Textfassung1-4von5-final.svg', 'M142_Textfassung1-1von2-final.svg', 'M141_Textfassung1-2von2-final.svg', 'M142_Textfassung1-2von2-final.svg', 'M141_Textfassung1-1von2-final.svg']

 Standard-Anker: M_138_TF1
   Zugeordnete SVGs: ['M138_Textfassung1-2von5-final.svg', 'M138_Textfassung1-1von5-final.svg', 'M138_Textfassung1-5von5-final.svg', 'M138_Textfassung1-4von5-final.svg', 'M138_Textfassung1-3von5-final.svg']
    g-tkk-1 -> g-tkk-1 (in M138_Textfassung1-1von5-final.svg)
    g-tkk-2 -> g-tkk-2 (in M138_Textfassung1-1von5-final.svg)
    g-tkk-3 -> g-tkk-3 (in M138_Textfassung1-1von5-final.svg)
    g-tkk-4 -> g-tkk-4 (in M138_Textfassung1-1von5-final.svg)
    g-tkk-5 ->

--- Starte Bearbeitung (Fuzzy-Logik deaktiviert für g-tkk- IDs) ---

[INFO] Block: op4_WE

[INFO] Block: M_138_TF1
    Update: g-tkk-49a -> g-tkk-50
    Update: g-tkk-50 -> g-tkk-51
    Update: g-tkk-51 -> g-tkk-52
    Update: g-tkk-52 -> g-tkk-53
    Update: g-tkk-53 -> g-tkk-54
    Update: g-tkk-54 -> g-tkk-55
    Update: g-tkk-55 -> g-tkk-56
    Update: g-tkk-56 -> g-tkk-57
    Update: g-tkk-57 -> g-tkk-58
    Update: g-tkk-58 -> g-tkk-59
    Update: g-tkk-59 -> g-tkk-60
    Update: g-tkk-60 -> g-tkk-61
    Update: g-tkk-61 -> g-tkk-62
    Update: g-tkk-62 -> g-tkk-63
    Update: g-tkk-63 -> g-tkk-64
    Update: g-tkk-64 -> g-tkk-65
    [LEER] ID 'g705d2' nicht zuordenbar (Fuzzy übersprungen: False)
    Update: g-tkk-65 -> g-tkk-66
    Update: g-tkk-66 -> g-tkk-67
    Update: g-tkk-67 -> g-tkk-68
    Update: g-tkk-68 -> g-tkk-69
    Update: g-tkk-69 -> g-tkk-70
    Update: g-tkk-70 -> g-tkk-71
    Update: g-tkk-71 -> g-tkk-72
    Update: g-tkk-72 -> g-tkk-73
    Update: g-tkk-73 -> 

In [ ]:
# made by Eli (lili041 --Github) with Google Gemini
# you have to fill in the  -path of .json textcritics-  and the  -path or .svg folder- !

# ACHTUNG: TODO werden übersprungen, das heisst aber auch, dass sie innerhalb eines Blockes nicht einberechnet werden, 
# und danach weitergezählt wird, als ob nichts wäre: g-tkk-1, g-tkk-2, g-tkk-3, g-tkk-4, TODO, g-tkk-5, g-tkk-6, ...


# --- KONFIGURATION ---


##### fill in:
json_path = '/Users/Elias/awg-app/src/assets/data/edition/series/1/section/5/op4/textcritics.json'

##### fill in:
svg_folder = '/Users/Elias/awg-app/src/assets/img/edition/series/1/section/5/op4' 


prefix = "g-tkk-"


import json
import os
import re

def extract_numbers(text):
    """Extrahiert Ziffern (z.B. 'M_143' -> '143')"""
    return "".join(re.findall(r'\d+', str(text)))

def display_uncertainties(data, prefix, loaded_svgs):
    """Checks JSON and SVG files for IDs that don't match the new prefix."""
    print(f"\n--- UNCERTAINTY & ERROR REPORT ---")
    errors_found = 0
    
    # 1. Check JSON for unchanged IDs
    all_entries = data.get('textcritics', data) if isinstance(data, dict) else data
    for entry in all_entries:
        entry_id = entry.get('id', 'Unknown')
        comments_list = entry.get('commentary', {}).get('comments', [])
        for comment_group in comments_list:
            for b_comment in comment_group.get('blockComments', []):
                val = b_comment.get('svgGroupId')
                if val and not val.startswith(prefix) and val != "TODO":
                    print(f"  [!] JSON ERROR: Unchanged ID '{val}' in Entry: {entry_id}")
                    errors_found += 1

    # 2. Check SVGs for IDs with class 'tkk' that don't match the prefix
    # Pattern looks for: id="anything" ... class="tkk" (or vice versa)
    tkk_pattern = re.compile(r'<[^>]*?id=["\'](.*?)["\'][^>]*?class=["\']tkk["\'][^>]*?>|<[^>]*?class=["\']tkk["\'][^>]*?id=["\'](.*?)["\'][^>]*?>')
    
    for filename, sdata in loaded_svgs.items():
        matches = tkk_pattern.findall(sdata["content"])
        for match in matches:
            # findall returns tuples based on groups; we grab the non-empty one
            found_id = match[0] if match[0] else match[1]
            if not found_id.startswith(prefix):
                print(f"  [!] SVG ORPHAN: ID '{found_id}' with class 'tkk' in {filename} was NOT updated.")
                errors_found += 1
    
    if errors_found == 0:
        print("  [✓] All JSON and SVG 'tkk' IDs successfully updated.")
    else:
        print(f"  [!] Total issues found: {errors_found}")

# 1. Daten laden
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

all_entries = data.get('textcritics', data) if isinstance(data, dict) else data
all_svg_files = [f for f in os.listdir(svg_folder) if f.endswith('.svg')]

# To keep track of all SVGs processed for the final report
final_svg_cache = {}
loaded_svg_texts = {}

def get_svg_text(filename):
    if filename not in loaded_svg_texts:
        path = os.path.join(svg_folder, filename)
        with open(path, 'r', encoding='utf-8') as f:
            content = f.read()
        loaded_svg_texts[filename] = {"content": content, "path": path}
        final_svg_cache[filename] = loaded_svg_texts[filename] # Store for final validation
    return loaded_svg_texts[filename]

current_main_number = None
relevant_svgs_for_block = []
id_mapping = {}

print(f"--- Starte Bearbeitung mit SkRT-Speziallogik ---")

# 2. Iteration durch die JSON
for entry in all_entries:
    if not isinstance(entry, dict): continue

    new_id = entry.get('id', '')
    if new_id:
        for fname, sdata in loaded_svg_texts.items():
            with open(sdata['path'], 'w', encoding='utf-8') as f:
                f.write(sdata['content'])
        
        loaded_svg_texts.clear()
        id_mapping.clear()
        
        current_main_number = extract_numbers(new_id)
        
        if "SkRT" in new_id:
            relevant_svgs_for_block = [
                f for f in all_svg_files 
                if current_main_number in extract_numbers(f) and "Reihentabelle" in f
            ]
            print(f"\n SkRT-Anker erkannt: {new_id}")
        else:
            relevant_svgs_for_block = [
                f for f in all_svg_files 
                if current_main_number in extract_numbers(f) and "Reihentabelle" not in f
            ]
            print(f"\n Standard-Anker: {new_id}")
            
        print(f"   Zugeordnete SVGs: {relevant_svgs_for_block}")

    comments_list = entry.get('commentary', {}).get('comments', [])
    for comment_group in comments_list:
        for b_comment in comment_group.get('blockComments', []):
            old_val = b_comment.get('svgGroupId')
            if not old_val or old_val == "TODO": continue

            pattern = rf'id=["\']{re.escape(old_val)}["\']'
            
            found_in_svg = None
            for svg_filename in relevant_svgs_for_block:
                svg_data = get_svg_text(svg_filename)
                if re.search(pattern, svg_data["content"]):
                    found_in_svg = svg_filename
                    break
            
            if found_in_svg:
                if old_val not in id_mapping:
                    id_mapping[old_val] = f"{prefix}{len(id_mapping) + 1}"
                
                new_val = id_mapping[old_val]
                
                print(f"    [JSON] Changing: '{old_val}' -> '{new_val}'")
                b_comment['svgGroupId'] = new_val
                
                svg_data = get_svg_text(found_in_svg)
                new_pattern_val = f'id="{new_val}"'
                print(f"    [SVG]  In {found_in_svg}: Replacing id=\"{old_val}\"")
                svg_data["content"] = re.sub(pattern, new_pattern_val, svg_data["content"])
            else:
                print(f"    [ERROR] ID '{old_val}' not found in any relevant SVGs for {new_id}")

# 3. Abschluss-Speicherung
for fname, sdata in loaded_svg_texts.items():
    with open(sdata['path'], 'w', encoding='utf-8') as f:
        f.write(sdata['content'])

# Final Validation Report
display_uncertainties(data, prefix, final_svg_cache)

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

print(f"\n Fertig!")
